# Notebook 02 — Baseline Visual Product Search

## Mục tiêu

Notebook này hiện thực hóa **baseline retrieval** sau khi Notebook 01 đã tạo:

```text
data/splits/sop_20k.csv
```

Baseline được cố định:

```text
Input image
    ↓
Resize
    ↓
Pretrained ResNet50
    ↓
Global Average Pooling
    ↓
L2 Normalization
    ↓
Exact Cosine Similarity Search
    ↓
Top-K retrieved products
```

Notebook được chia thành 2 phần:

### Offline

```text
Gallery images
    → preprocessing
    → ResNet50
    → embedding
    → L2
    → lưu gallery embeddings
```

### Online

```text
Query image
    → preprocessing
    → ResNet50
    → embedding
    → L2
    → exact cosine search trên gallery embeddings
    → Top-K
```

Sau đó notebook đo:

- Recall@1
- Recall@5
- Recall@10
- Recall@20
- Recall@50
- Recall@100
- mAP
- offline embedding time
- online query latency
- embedding/index memory
- qualitative retrieval

> **Lưu ý:** Notebook này không dùng DINOv3, HNSW hoặc metadata re-ranking. Đây là baseline thuần để làm mốc so sánh với proposed.

## Cell 1 — Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# Input từ Notebook 01
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
SAMPLE_FILE = SPLIT_DIR / "sop_20k.csv"

# Output baseline
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = OUTPUT_DIR / "gallery_embeddings.npy"
GALLERY_META_FILE = OUTPUT_DIR / "gallery_metadata.csv"
QUERY_RESULTS_FILE = OUTPUT_DIR / "retrieval_results.npy"
METRICS_FILE = OUTPUT_DIR / "metrics.json"
LATENCY_FILE = OUTPUT_DIR / "latency.json"

# Model
MODEL_NAME = "resnet50"
PRETRAINED = True

# Input
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# Retrieval
KS = [1, 5, 10, 20, 50, 100]

# Evaluation
MAX_QUERIES = None       # None = toàn bộ query
RANDOM_SEED = 42

# Device
DEVICE = "cuda"          # đổi thành "cpu" nếu cần

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SAMPLE_FILE :", SAMPLE_FILE)
print("OUTPUT_DIR  :", OUTPUT_DIR)

## Cell 2 — Imports

In [ ]:
import os
import sys
import json
import time
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import models, transforms

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 3 — Kiểm tra input từ Notebook 01

In [ ]:
if not SAMPLE_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {SAMPLE_FILE}. "
        "Hãy chạy Notebook 01 trước."
    )

sample_df = pd.read_csv(SAMPLE_FILE)

print("Rows:", len(sample_df))
print("Classes:", sample_df["class_id"].nunique())

display(sample_df.head())

## Cell 4 — Kiểm tra image paths

In [ ]:
sample_df["exists"] = sample_df["image_path"].map(os.path.exists)

print("Existing:", sample_df["exists"].sum())
print("Missing :", (~sample_df["exists"]).sum())

if not sample_df["exists"].all():
    display(
        sample_df.loc[
            ~sample_df["exists"],
            ["image_id", "class_id", "image_path"]
        ].head(20)
    )

    raise FileNotFoundError(
        "Có image path không tồn tại."
    )

print("✓ All image paths exist.")

## Cell 5 — Tạo query/gallery split cho baseline

Baseline cần một **gallery** để search và một **query set** để evaluation.

Split này được tạo **sau sampling 20K**, không sampling lại từ 120K.

Với mỗi `class_id`:

- 80% → gallery
- 20% → query
- tối thiểu 1 query
- tối thiểu 1 gallery image

Điều này đảm bảo query có positive image trong gallery.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

gallery_parts = []
query_parts = []

for class_id, group in sample_df.groupby("class_id"):
    group = group.sample(
        frac=1,
        random_state=RANDOM_SEED + int(class_id) % 100000
    )

    n = len(group)

    n_query = max(1, int(round(n * 0.20)))
    n_query = min(n_query, n - 1)

    query_parts.append(group.iloc[:n_query])
    gallery_parts.append(group.iloc[n_query:])

query_df = pd.concat(
    query_parts,
    ignore_index=True
)

gallery_df = pd.concat(
    gallery_parts,
    ignore_index=True
)

print("Gallery:", len(gallery_df))
print("Query  :", len(query_df))

print("Gallery classes:", gallery_df.class_id.nunique())
print("Query classes  :", query_df.class_id.nunique())

missing_query_classes = (
    set(query_df.class_id)
    - set(gallery_df.class_id)
)

print(
    "Query classes missing in gallery:",
    len(missing_query_classes)
)

assert len(missing_query_classes) == 0

## Cell 6 — Lưu query/gallery split

In [ ]:
gallery_file = SPLIT_DIR / "baseline_gallery.csv"
query_file = SPLIT_DIR / "baseline_query.csv"

gallery_df.to_csv(
    gallery_file,
    index=False
)

query_df.to_csv(
    query_file,
    index=False
)

print("Saved:", gallery_file)
print("Saved:", query_file)

## Cell 7 — Kiểm tra split distribution

In [ ]:
split_stats = pd.DataFrame({
    "gallery": gallery_df["class_id"].value_counts(),
    "query": query_df["class_id"].value_counts(),
}).fillna(0)

split_stats["total"] = (
    split_stats["gallery"]
    + split_stats["query"]
)

display(split_stats.head(20))

print("Gallery ratio:", len(gallery_df) / len(sample_df))
print("Query ratio  :", len(query_df) / len(sample_df))

## Cell 8 — Baseline preprocessing

Baseline chỉ dùng preprocessing tối thiểu:

```text
PIL image
 → RGB
 → Resize(224,224)
 → ToTensor
 → ImageNet normalization
```

Không sử dụng:

- object detection
- segmentation
- background removal
- illumination correction
- DINO-specific preprocessing
- metadata

In [ ]:
IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

baseline_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
])

print(baseline_transform)

## Cell 9 — Preview preprocessing

In [ ]:
row = gallery_df.iloc[0]

original = Image.open(
    row["image_path"]
).convert("RGB")

processed_tensor = baseline_transform(original)

# Unnormalize for visualization
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

processed_image = (
    processed_tensor.cpu() * std + mean
).clamp(0, 1)

plt.figure(figsize=(9, 4))

ax = plt.subplot(1, 2, 1)
ax.imshow(original)
ax.set_title("Original")
ax.axis("off")

ax = plt.subplot(1, 2, 2)
ax.imshow(processed_image.permute(1, 2, 0))
ax.set_title("Baseline input: 224x224")
ax.axis("off")

plt.tight_layout()
plt.show()

## Cell 10 — Dataset class

In [ ]:
class SOPImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "image_id": int(row["image_id"]),
            "class_id": int(row["class_id"]),
            "index": idx,
        }

## Cell 11 — DataLoader

In [ ]:
gallery_dataset = SOPImageDataset(
    gallery_df,
    transform=baseline_transform
)

query_dataset = SOPImageDataset(
    query_df,
    transform=baseline_transform
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

query_loader = DataLoader(
    query_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("Gallery batches:",
      len(gallery_loader))

print("Query batches:",
      len(query_loader))

## Cell 12 — Load pretrained ResNet50

Baseline encoder:

```text
ResNet50 pretrained on ImageNet
        ↓
remove classification FC
        ↓
2048-dimensional feature
```

Vì `avgpool` nằm ngay trước `fc`, output sau `avgpool` là:

```text
[B, 2048, 1, 1]
```

sau flatten:

```text
[B, 2048]
```

In [ ]:
weights = (
    models.ResNet50_Weights.DEFAULT
    if PRETRAINED
    else None
)

resnet = models.resnet50(
    weights=weights
)

# Remove classification head
encoder = nn.Sequential(
    *list(resnet.children())[:-1]
)

encoder = encoder.to(DEVICE)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

print(encoder)

## Cell 13 — Kiểm tra feature dimension

In [ ]:
dummy = torch.randn(
    2, 3,
    IMAGE_SIZE,
    IMAGE_SIZE
).to(DEVICE)

with torch.inference_mode():
    feature = encoder(dummy)

print("Raw output shape:", feature.shape)

feature = torch.flatten(
    feature,
    start_dim=1
)

print("Flattened feature shape:", feature.shape)

FEATURE_DIM = feature.shape[1]

assert FEATURE_DIM == 2048

print("Feature dimension:", FEATURE_DIM)

## Cell 14 — Hàm L2 normalization

In [ ]:
def l2_normalize(x, eps=1e-12):
    """
    Row-wise L2 normalization.
    x: [N, D]
    """
    norm = torch.linalg.vector_norm(
        x,
        ord=2,
        dim=1,
        keepdim=True
    )

    return x / norm.clamp_min(eps)


test_x = torch.randn(4, FEATURE_DIM).to(DEVICE)
test_y = l2_normalize(test_x)

norms = torch.linalg.vector_norm(
    test_y,
    ord=2,
    dim=1
)

print(norms)

assert torch.allclose(
    norms,
    torch.ones_like(norms),
    atol=1e-5
)

print("✓ L2 normalization correct.")

## Cell 15 — Hàm extract embeddings

Đây là **offline feature extraction**.

Input:

```text
gallery images
```

Output:

```text
gallery_embeddings.npy
```

Shape:

```text
[num_gallery_images, 2048]
```

In [ ]:
def extract_embeddings(
    model,
    dataloader,
    device,
):
    model.eval()

    all_embeddings = []
    all_image_ids = []
    all_class_ids = []

    start = time.perf_counter()

    with torch.inference_mode():

        for batch in tqdm(
            dataloader,
            desc="Extract embeddings"
        ):
            images = batch["image"].to(
                device,
                non_blocking=True
            )

            features = model(images)

            features = torch.flatten(
                features,
                start_dim=1
            )

            features = l2_normalize(
                features
            )

            all_embeddings.append(
                features.cpu().numpy()
            )

            all_image_ids.extend(
                batch["image_id"].numpy()
            )

            all_class_ids.extend(
                batch["class_id"].numpy()
            )

    elapsed = time.perf_counter() - start

    embeddings = np.concatenate(
        all_embeddings,
        axis=0
    ).astype(np.float32)

    return (
        embeddings,
        np.asarray(all_image_ids),
        np.asarray(all_class_ids),
        elapsed
    )

## Cell 16 — Offline: extract gallery embeddings

In [ ]:
if DEVICE == "cuda":
    torch.cuda.synchronize()

gallery_embeddings, gallery_image_ids, gallery_class_ids, gallery_extract_time = \
    extract_embeddings(
        encoder,
        gallery_loader,
        DEVICE
    )

if DEVICE == "cuda":
    torch.cuda.synchronize()

print("Embedding shape:", gallery_embeddings.shape)
print("Extraction time:", gallery_extract_time, "sec")
print("dtype:", gallery_embeddings.dtype)

## Cell 17 — Validate gallery embeddings

In [ ]:
assert gallery_embeddings.ndim == 2
assert gallery_embeddings.shape[0] == len(gallery_df)
assert gallery_embeddings.shape[1] == 2048

embedding_norms = np.linalg.norm(
    gallery_embeddings,
    axis=1
)

print(
    "Norm min:",
    embedding_norms.min()
)

print(
    "Norm max:",
    embedding_norms.max()
)

print(
    "Norm mean:",
    embedding_norms.mean()
)

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-4
)

print("✓ Gallery embeddings are L2 normalized.")

## Cell 18 — Lưu offline artifacts

In [ ]:
np.save(
    EMBEDDING_FILE,
    gallery_embeddings
)

gallery_meta = gallery_df.copy()

gallery_meta.to_csv(
    GALLERY_META_FILE,
    index=False
)

print("Embedding saved:", EMBEDDING_FILE)
print("Metadata saved :", GALLERY_META_FILE)

## Cell 19 — Đo memory của embedding

In [ ]:
embedding_memory_bytes = (
    gallery_embeddings.nbytes
)

embedding_memory_mb = (
    embedding_memory_bytes
    / (1024 ** 2)
)

print(
    f"Gallery embedding memory: "
    f"{embedding_memory_mb:.2f} MB"
)

print(
    f"Per vector: "
    f"{gallery_embeddings.shape[1] * 4 / 1024:.2f} KB"
)

## Cell 20 — Exact cosine search

Vì tất cả embedding đã được L2 normalize:

```text
cosine(q, x)
=
q · x
```

Do đó exact cosine search có thể thực hiện bằng matrix multiplication:

```text
Q [Nq × D]
×
Gᵀ [D × Ng]
=
S [Nq × Ng]
```

Sau đó lấy Top-K similarity lớn nhất.

Đây là **exact search**, không dùng ANN.

In [ ]:
def exact_cosine_search(
    query_embeddings,
    gallery_embeddings,
    top_k=100,
):
    """
    Exact cosine search.

    Both query and gallery embeddings
    must be L2 normalized.
    """

    similarities = (
        query_embeddings
        @ gallery_embeddings.T
    )

    top_k = min(
        top_k,
        gallery_embeddings.shape[0]
    )

    # Partial sort
    candidate = np.argpartition(
        -similarities,
        kth=top_k - 1,
        axis=1
    )[:, :top_k]

    candidate_scores = np.take_along_axis(
        similarities,
        candidate,
        axis=1
    )

    # Sort candidate results
    order = np.argsort(
        -candidate_scores,
        axis=1
    )

    indices = np.take_along_axis(
        candidate,
        order,
        axis=1
    )

    scores = np.take_along_axis(
        candidate_scores,
        order,
        axis=1
    )

    return indices, scores

## Cell 21 — Test exact search với 1 query

In [ ]:
test_query = gallery_embeddings[
    0:1
]

indices, scores = exact_cosine_search(
    test_query,
    gallery_embeddings,
    top_k=5
)

print("Top indices:", indices)
print("Scores:", scores)

print(
    "Self retrieval:",
    indices[0, 0] == 0
)

assert indices[0, 0] == 0
print("✓ Exact cosine search works.")

## Cell 22 — Offline → Online boundary

Từ đây:

### Offline đã hoàn thành

```text
Gallery
 → ResNet50
 → L2
 → gallery_embeddings.npy
```

Online chỉ cần:

```text
Query image
 → ResNet50
 → L2
 → exact search trên gallery_embeddings
```

**Không chạy ResNet50 lại cho gallery.**

## Cell 23 — Hàm encode query

In [ ]:
def encode_query_image(
    image_path,
    model,
    transform,
    device,
):
    image = Image.open(
        image_path
    ).convert("RGB")

    tensor = transform(image)
    tensor = tensor.unsqueeze(0)
    tensor = tensor.to(device)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        feature = model(tensor)

        feature = torch.flatten(
            feature,
            start_dim=1
        )

        feature = l2_normalize(
            feature
        )

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    return (
        feature.cpu().numpy(),
        elapsed
    )

## Cell 24 — Online retrieval cho một query

In [ ]:
QUERY_INDEX = 0
ONLINE_TOP_K = 100

query_row = query_df.iloc[QUERY_INDEX]

query_embedding, query_encode_time = \
    encode_query_image(
        query_row["image_path"],
        encoder,
        baseline_transform,
        DEVICE
    )

search_start = time.perf_counter()

retrieved_indices, retrieved_scores = \
    exact_cosine_search(
        query_embedding,
        gallery_embeddings,
        top_k=ONLINE_TOP_K
    )

search_time = (
    time.perf_counter()
    - search_start
)

retrieved_indices = retrieved_indices[0]
retrieved_scores = retrieved_scores[0]

print("Query:", query_row["image_id"])
print("GT class:", query_row["class_id"])
print("Encoding:", query_encode_time * 1000, "ms")
print("Search  :", search_time * 1000, "ms")

## Cell 25 — Xem Top-10 kết quả

In [ ]:
rows = []

for rank, (idx, score) in enumerate(
    zip(
        retrieved_indices[:10],
        retrieved_scores[:10]
    ),
    start=1
):
    item = gallery_df.iloc[int(idx)]

    rows.append({
        "rank": rank,
        "gallery_index": int(idx),
        "image_id": int(item["image_id"]),
        "class_id": int(item["class_id"]),
        "similarity": float(score),
        "correct": (
            int(item["class_id"])
            == int(query_row["class_id"])
        ),
        "image_path": item["image_path"],
    })

top10_df = pd.DataFrame(rows)

display(top10_df)

## Cell 26 — Hàm Recall@K và Average Precision

In [ ]:
def recall_at_k(
    retrieved_class_ids,
    ground_truth_class_id,
    k
):
    retrieved = np.asarray(
        retrieved_class_ids[:k]
    )

    return float(
        np.any(
            retrieved
            == ground_truth_class_id
        )
    )


def average_precision(
    retrieved_class_ids,
    ground_truth_class_id
):
    """
    AP for instance/product retrieval
    where every gallery image having the
    same class_id is considered relevant.
    """

    retrieved = np.asarray(
        retrieved_class_ids
    )

    relevant = (
        retrieved
        == ground_truth_class_id
    )

    num_relevant = relevant.sum()

    if num_relevant == 0:
        return 0.0

    precisions = []

    hits = 0

    for rank, is_relevant in enumerate(
        relevant,
        start=1
    ):
        if is_relevant:
            hits += 1
            precisions.append(
                hits / rank
            )

    return float(
        np.mean(precisions)
    )

## Cell 27 — Offline query embedding extraction

In [ ]:
if MAX_QUERIES is None:
    eval_query_df = query_df.copy()
else:
    eval_query_df = query_df.head(
        MAX_QUERIES
    ).copy()

eval_query_dataset = SOPImageDataset(
    eval_query_df,
    transform=baseline_transform
)

eval_query_loader = DataLoader(
    eval_query_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

query_embeddings, query_image_ids, query_class_ids, query_extract_time = \
    extract_embeddings(
        encoder,
        eval_query_loader,
        DEVICE
    )

print(
    "Query embedding shape:",
    query_embeddings.shape
)

print(
    "Query extraction time:",
    query_extract_time,
    "sec"
)

## Cell 28 — Exact retrieval toàn bộ query

In [ ]:
if DEVICE == "cuda":
    torch.cuda.synchronize()

search_start = time.perf_counter()

retrieval_indices, retrieval_scores = \
    exact_cosine_search(
        query_embeddings,
        gallery_embeddings,
        top_k=max(KS)
    )

if DEVICE == "cuda":
    torch.cuda.synchronize()

total_search_time = (
    time.perf_counter()
    - search_start
)

num_eval_queries = len(eval_query_df)

avg_search_ms = (
    total_search_time
    / num_eval_queries
    * 1000
)

print(
    "Queries:",
    num_eval_queries
)

print(
    "Total exact search:",
    total_search_time,
    "sec"
)

print(
    "Average search/query:",
    avg_search_ms,
    "ms"
)

## Cell 29 — Tính Recall@K và mAP

In [ ]:
retrieved_class_matrix = (
    gallery_df.iloc[
        retrieval_indices.reshape(-1)
    ]["class_id"]
    .to_numpy()
    .reshape(
        retrieval_indices.shape
    )
)

metrics = {}

for k in KS:

    recalls = []

    for i in range(
        num_eval_queries
    ):
        recalls.append(
            recall_at_k(
                retrieved_class_matrix[i],
                query_class_ids[i],
                k
            )
        )

    metrics[f"Recall@{k}"] = float(
        np.mean(recalls)
    )

aps = []

for i in range(
    num_eval_queries
):
    aps.append(
        average_precision(
            retrieved_class_matrix[i],
            query_class_ids[i]
        )
    )

metrics["mAP"] = float(
    np.mean(aps)
)

print(json.dumps(
    metrics,
    indent=2
))

## Cell 30 — Kiểm tra Recall@K có tính đơn điệu

In [ ]:
recall_values = [
    metrics[f"Recall@{k}"]
    for k in KS
]

print(
    list(zip(KS, recall_values))
)

for a, b in zip(
    recall_values,
    recall_values[1:]
):
    assert b + 1e-12 >= a

print(
    "✓ Recall@K is non-decreasing."
)

## Cell 31 — Phân tích latency

Tách latency thành:

```text
1. Query encoding
2. Exact search
3. End-to-end
```

Đây là quan trọng khi so sánh với proposed vì proposed sẽ có:

```text
query preprocessing
→ DINOv3
→ HNSW
→ re-ranking
```

In [ ]:
# Measure per-query encoding + search
encoding_times = []
search_times = []
end_to_end_times = []

for _, row in tqdm(
    eval_query_df.iterrows(),
    total=len(eval_query_df),
    desc="Latency benchmark"
):

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start_total = time.perf_counter()

    embedding, enc_time = \
        encode_query_image(
            row["image_path"],
            encoder,
            baseline_transform,
            DEVICE
        )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start_search = time.perf_counter()

    exact_cosine_search(
        embedding,
        gallery_embeddings,
        top_k=max(KS)
    )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    end_total = time.perf_counter()

    encoding_times.append(
        enc_time
    )

    search_times.append(
        end_total - start_search
    )

    end_to_end_times.append(
        end_total - start_total
    )

latency_stats = {
    "encoding_mean_ms":
        np.mean(encoding_times) * 1000,

    "encoding_p50_ms":
        np.percentile(
            encoding_times, 50
        ) * 1000,

    "encoding_p95_ms":
        np.percentile(
            encoding_times, 95
        ) * 1000,

    "search_mean_ms":
        np.mean(search_times) * 1000,

    "search_p50_ms":
        np.percentile(
            search_times, 50
        ) * 1000,

    "search_p95_ms":
        np.percentile(
            search_times, 95
        ) * 1000,

    "end_to_end_mean_ms":
        np.mean(end_to_end_times) * 1000,

    "end_to_end_p50_ms":
        np.percentile(
            end_to_end_times, 50
        ) * 1000,

    "end_to_end_p95_ms":
        np.percentile(
            end_to_end_times, 95
        ) * 1000,
}

print(json.dumps(
    latency_stats,
    indent=2
))

## Cell 32 — Đo memory

In [ ]:
# Main stored arrays
gallery_embedding_mb = (
    gallery_embeddings.nbytes
    / 1024**2
)

query_embedding_mb = (
    query_embeddings.nbytes
    / 1024**2
)

# Exact search does not create a persistent index.
# Similarity matrix can be large; for this notebook
# we only report the theoretical full matrix size.
similarity_matrix_bytes = (
    len(query_df)
    * len(gallery_df)
    * 4
)

similarity_matrix_mb = (
    similarity_matrix_bytes
    / 1024**2
)

memory_stats = {
    "gallery_embeddings_MB":
        gallery_embedding_mb,

    "query_embeddings_MB":
        query_embedding_mb,

    "theoretical_full_similarity_matrix_MB":
        similarity_matrix_mb,

    "feature_dimension":
        int(FEATURE_DIM),

    "gallery_size":
        int(len(gallery_df)),
}

print(json.dumps(
    memory_stats,
    indent=2
))

## Cell 33 — Lưu retrieval results

In [ ]:
np.save(
    QUERY_RESULTS_FILE,
    retrieval_indices
)

np.save(
    OUTPUT_DIR / "retrieval_scores.npy",
    retrieval_scores
)

print(
    "Saved:",
    QUERY_RESULTS_FILE
)

## Cell 34 — Lưu metrics

In [ ]:
final_metrics = {
    **metrics,

    "num_gallery":
        int(len(gallery_df)),

    "num_queries":
        int(num_eval_queries),

    "feature_dim":
        int(FEATURE_DIM),

    "gallery_embedding_MB":
        float(gallery_embedding_mb),

    "gallery_extraction_time_sec":
        float(gallery_extract_time),

    "query_extraction_time_sec":
        float(query_extract_time),

    "exact_search_total_sec":
        float(total_search_time),

    "exact_search_avg_ms":
        float(avg_search_ms),

    **latency_stats,
}

with open(
    METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2
    )

with open(
    LATENCY_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        latency_stats,
        f,
        indent=2
    )

print(json.dumps(
    final_metrics,
    indent=2
))

## Cell 35 — Plot Recall@K

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    KS,
    [
        metrics[f"Recall@{k}"]
        for k in KS
    ],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("Recall@K")
plt.title("Baseline Recall@K")
plt.xticks(KS)
plt.grid(alpha=0.2)

recall_plot = (
    OUTPUT_DIR / "recall_at_k.png"
)

plt.savefig(
    recall_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", recall_plot)

## Cell 36 — Chọn một query để qualitative visualization

In [ ]:
QUAL_QUERY_INDEX = 0
QUAL_TOP_K = 5

q = eval_query_df.iloc[
    QUAL_QUERY_INDEX
]

q_embedding = query_embeddings[
    QUAL_QUERY_INDEX:
    QUAL_QUERY_INDEX + 1
]

q_indices, q_scores = \
    exact_cosine_search(
        q_embedding,
        gallery_embeddings,
        top_k=QUAL_TOP_K
    )

q_indices = q_indices[0]
q_scores = q_scores[0]

print("Query image:", q["image_path"])
print("GT class:", q["class_id"])

qual_rows = []

for rank, (idx, score) in enumerate(
    zip(q_indices, q_scores),
    start=1
):

    g = gallery_df.iloc[int(idx)]

    qual_rows.append({
        "rank": rank,
        "image_id": int(g["image_id"]),
        "class_id": int(g["class_id"]),
        "similarity": float(score),
        "correct": (
            int(g["class_id"])
            == int(q["class_id"])
        ),
        "image_path": g["image_path"],
    })

qual_df = pd.DataFrame(
    qual_rows
)

display(qual_df)

## Cell 37 — Vẽ query + Top-K

In [ ]:
def show_baseline_retrieval(
    query_row,
    result_df,
    save_path=None
):

    n = len(result_df)

    plt.figure(
        figsize=(3 * (n + 1), 4)
    )

    # Query
    ax = plt.subplot(
        1, n + 1, 1
    )

    ax.imshow(
        Image.open(
            query_row["image_path"]
        ).convert("RGB")
    )

    ax.set_title(
        f"QUERY\nclass={query_row['class_id']}"
    )

    ax.axis("off")

    # Results
    for i, (_, row) in enumerate(
        result_df.iterrows(),
        start=1
    ):

        ax = plt.subplot(
            1, n + 1, i + 1
        )

        ax.imshow(
            Image.open(
                row["image_path"]
            ).convert("RGB")
        )

        correct_text = (
            "✓" if row["correct"]
            else "✗"
        )

        ax.set_title(
            f"Top-{row['rank']} {correct_text}\n"
            f"class={row['class_id']}\n"
            f"sim={row['similarity']:.3f}"
        )

        ax.axis("off")

    plt.suptitle(
        "Baseline: ResNet50 + Exact Cosine"
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            dpi=200,
            bbox_inches="tight"
        )

    plt.show()


qual_path = (
    OUTPUT_DIR
    / "qualitative_query_0000.png"
)

show_baseline_retrieval(
    q,
    qual_df,
    save_path=qual_path
)

print("Saved:", qual_path)

## Cell 38 — Xuất nhiều qualitative cases

In [ ]:
QUAL_DIR = OUTPUT_DIR / "qualitative"
QUAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

N_QUALITATIVE = min(
    20,
    len(eval_query_df)
)

for qi in tqdm(
    range(N_QUALITATIVE),
    desc="Export qualitative examples"
):

    q = eval_query_df.iloc[qi]

    q_embedding = query_embeddings[
        qi:qi + 1
    ]

    indices, scores = \
        exact_cosine_search(
            q_embedding,
            gallery_embeddings,
            top_k=QUAL_TOP_K
        )

    indices = indices[0]
    scores = scores[0]

    rows = []

    for rank, (idx, score) in enumerate(
        zip(indices, scores),
        start=1
    ):

        g = gallery_df.iloc[int(idx)]

        rows.append({
            "rank": rank,
            "class_id": int(g["class_id"]),
            "similarity": float(score),
            "correct": (
                int(g["class_id"])
                == int(q["class_id"])
            ),
            "image_path": g["image_path"],
        })

    result_df = pd.DataFrame(rows)

    save_path = (
        QUAL_DIR
        / f"query_{qi:04d}.png"
    )

    show_baseline_retrieval(
        q,
        result_df,
        save_path=save_path
    )

    plt.close("all")

print(
    "Qualitative directory:",
    QUAL_DIR
)

## Cell 39 — Tổng hợp qualitative accuracy

In [ ]:
qualitative_summary = []

for qi in range(
    min(N_QUALITATIVE, len(eval_query_df))
):

    q = eval_query_df.iloc[qi]

    q_embedding = query_embeddings[
        qi:qi + 1
    ]

    indices, _ = exact_cosine_search(
        q_embedding,
        gallery_embeddings,
        top_k=QUAL_TOP_K
    )

    retrieved_classes = (
        gallery_df.iloc[
            indices[0]
        ]["class_id"].to_numpy()
    )

    qualitative_summary.append({
        "query_index": qi,
        "query_image_id": int(q["image_id"]),
        "query_class_id": int(q["class_id"]),
        "top1_correct": (
            retrieved_classes[0]
            == q["class_id"]
        ),
        "top5_correct_count": int(
            np.sum(
                retrieved_classes
                == q["class_id"]
            )
        ),
    })

qualitative_summary_df = pd.DataFrame(
    qualitative_summary
)

display(
    qualitative_summary_df
)

qualitative_summary_df.to_csv(
    OUTPUT_DIR
    / "qualitative_summary.csv",
    index=False
)

## Cell 40 — Final baseline report

In [ ]:
print("=" * 70)
print("BASELINE EXPERIMENT COMPLETED")
print("=" * 70)

print("\nDataset")
print("Gallery:", len(gallery_df))
print("Query  :", len(eval_query_df))
print("Classes:", sample_df.class_id.nunique())

print("\nModel")
print("Encoder:", MODEL_NAME)
print("Pretrained:", PRETRAINED)
print("Feature dimension:", FEATURE_DIM)

print("\nRetrieval")
print("Method: Exact cosine")
print("Top-K:", max(KS))

print("\nMetrics")
for k in KS:
    print(
        f"Recall@{k}: "
        f"{metrics[f'Recall@{k}']:.4f}"
    )

print(
    f"mAP: {metrics['mAP']:.4f}"
)

print("\nLatency")
print(
    f"Query encoding mean: "
    f"{latency_stats['encoding_mean_ms']:.2f} ms"
)

print(
    f"Exact search mean: "
    f"{latency_stats['search_mean_ms']:.2f} ms"
)

print(
    f"End-to-end mean: "
    f"{latency_stats['end_to_end_mean_ms']:.2f} ms"
)

print("\nMemory")
print(
    f"Gallery embeddings: "
    f"{gallery_embedding_mb:.2f} MB"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

# Output expected

Sau khi chạy xong Notebook 02:

```text
outputs/
└── baseline/
    ├── gallery_embeddings.npy
    ├── gallery_metadata.csv
    ├── retrieval_results.npy
    ├── retrieval_scores.npy
    ├── metrics.json
    ├── latency.json
    ├── recall_at_k.png
    ├── qualitative_query_0000.png
    ├── qualitative_summary.csv
    │
    └── qualitative/
        ├── query_0000.png
        ├── query_0001.png
        ├── ...
```

và:

```text
data/
└── splits/
    ├── sop_20k.csv
    ├── baseline_gallery.csv
    └── baseline_query.csv
```

## Baseline cuối cùng

```text
                 OFFLINE
Gallery ──→ Resize 224
              ↓
          ResNet50
              ↓
             GAP
              ↓
          L2 norm
              ↓
       gallery embeddings
              │
              │ save
              ▼
        embeddings.npy


                  ONLINE
Query image ─→ Resize 224
                  ↓
              ResNet50
                  ↓
                 GAP
                  ↓
               L2 norm
                  ↓
        Exact cosine search
                  ↓
                Top-K
```

Notebook 03 sẽ giữ **cùng query/gallery split và evaluation protocol**, sau đó thay encoder/retrieval bằng proposed:

```text
preprocessing
    ↓
DINOv3
    ↓
L2
    ↓
FAISS HNSW
    ↓
metadata re-ranking
    ↓
Top-K
```

Như vậy `02_baseline` và `03_proposed` có thể so sánh trực tiếp Recall@K, mAP, latency và memory.